In [1]:
import os
import math
import glob
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from torch.optim import Adam
from torchinfo import summary
from torchvision.transforms import v2
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
HR_train_paths = sorted(glob.glob("../data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("../data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X64/*.png"))

## Model and data preparation

Calculate mean RGB values of DIV2K dataset for input normalization

In [4]:
# R, G, B = 0, 0, 0
# total_pixels = 0
# for path in tqdm(HR_train_paths):
#     img = Image.open(path).convert('RGB')
#     img_np = np.array(img) / 255.0

#     h, w, _ = img_np.shape
#     total_pixels += h * w
    
#     R += np.sum(img_np[:, :, 0])
#     G += np.sum(img_np[:, :, 1])
#     B += np.sum(img_np[:, :, 2])

# R /= total_pixels
# G /= total_pixels
# B /= total_pixels
# R, G, B

### RCAN

The architecture below is described in this [paper](https://arxiv.org/abs/1807.02758)

In [5]:
class RCAB(nn.Module):
    """
    Residual Channel Attention Block
    """
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding='same')
        )

        self.attention_block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(64, 64//16, 1),
            nn.ReLU(),
            nn.Conv2d(64//16, 64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        Xgb = self.block1(x)
        return x + Xgb * self.attention_block(Xgb)
    
class RG(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential()

        for _ in range(20):
            self.block.append(RCAB())

        self.block.append(nn.Conv2d(64, 64, 3, padding='same'))

    def forward(self, x):
        return x + self.block(x)
    
class RIR(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential()

        for _ in range(10):
            self.block.append(RG())

        self.block.append(nn.Conv2d(64, 64, 3, padding='same'))

    def forward(self, x):
        return x + self.block(x)

class RCAN(nn.Module):
    def __init__(self, n):
        """
        Args:
            n: scaling factor
        """
        super().__init__()
        self.DIV2K_RGB = torch.tensor([0.44882884613943946, 0.43713809810624193, 0.4040371984052683]).view(1, 3, 1, 1).to(device)

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding='same'),
            RIR()
        )

        self.upscaling_head = nn.Sequential()
        for _ in range(int(math.log2(n))):
            self.upscaling_head.append(nn.Conv2d(64, 4*64, 3, padding='same'))
            self.upscaling_head.append(nn.PixelShuffle(2))
            self.upscaling_head.append(nn.PReLU())
            
        self.upscaling_head.append(nn.Conv2d(64, 3, 3, padding='same'))

    def forward(self, x):
        return self.upscaling_head(self.feature_extractor(x-self.DIV2K_RGB)) + self.DIV2K_RGB

In [6]:
summary(RCAN(8), input_size=(16, 3, 48, 48))

Layer (type:depth-idx)                                                 Output Shape              Param #
RCAN                                                                   [16, 3, 384, 384]         --
├─Sequential: 1-1                                                      [16, 64, 48, 48]          --
│    └─Conv2d: 2-1                                                     [16, 64, 48, 48]          1,792
│    └─RIR: 2-2                                                        [16, 64, 48, 48]          --
│    │    └─Sequential: 3-1                                            [16, 64, 48, 48]          15,293,408
├─Sequential: 1-2                                                      [16, 3, 384, 384]         --
│    └─Conv2d: 2-3                                                     [16, 256, 48, 48]         147,712
│    └─PixelShuffle: 2-4                                               [16, 64, 96, 96]          --
│    └─PReLU: 2-5                                                      [16, 64,

In [7]:
class RCAN_Dataset(Dataset):
    def __init__(self, target_paths: list[str], scale: int, ram_limit_gb: float = 2.0):
        self.crop_size = scale * 48
        self.scale = scale

        self.rotations = [0, 90, 180, 270]
        self.transforms = v2.Compose([
            v2.PILToTensor(),
            v2.Lambda(lambda x: (x / 255.0))
        ])

        self.preloaded = {}
        self.paths = target_paths

        total_ram_used = 0
        for i, path in enumerate(tqdm(target_paths, desc="Preloading images")):
            img = Image.open(path).convert("RGB")
            total_ram_used += img.width * img.height * 3 / (1024 ** 3)  # ~size in GB

            if total_ram_used < ram_limit_gb:
                self.preloaded[i] = img
            else:
                break

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        if idx in self.preloaded:
            target = self.preloaded[idx]
        else:
            target = Image.open(self.paths[idx]).convert("RGB")

        target = self.random_crop(target, self.crop_size)
        inp = target.resize((target.width // self.scale, target.height // self.scale), Image.BICUBIC)
        
        rotation =  random.choice(self.rotations)
        if rotation != 0:
            inp = v2.functional.rotate(inp, rotation)
            target = v2.functional.rotate(target, rotation)
        if random.randint(0, 1):
            inp = v2.functional.horizontal_flip(inp)
            target = v2.functional.horizontal_flip(target)
            
        return self.transforms(inp), self.transforms(target)

    def random_crop(self, img, size):
        w, h = img.size
        if w < size or h < size:
            img = img.resize((size, size), Image.BICUBIC)
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)
        return img.crop((x, y, x + size, y + size))

    def set_scale(self, scale: int):
        self.scale = scale

    def set_crop_size(self, crop_size: int):
        self.crop_size = crop_size

In [8]:
psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips = LearnedPerceptualImagePatchSimilarity(normalize=True).to(device)

In [9]:
transform = v2.Compose([
    v2.PILToTensor(),
    v2.Lambda(lambda x: x / 255.0)
])

In [10]:
def calc_metrics(model: nn.Module, target_ds: list[str], scale: int):
    transform_target = v2.Compose([
        v2.PILToTensor(),
        v2.Lambda(lambda x: x/255.0)
    ])

    transform_input = v2.Compose([
        v2.PILToTensor(),
        v2.Lambda(lambda x: (x / 255.0))
    ])

    psnr_acc = 0
    ssim_acc = 0
    lpips_acc = 0
    failed_lpips = 0

    for i in tqdm(range(len(target_ds)), leave=False):
        target_image = Image.open(target_ds[i]).convert("RGB")
        w, h = target_image.size

        w -= w % scale
        h -= h % scale
        target_image = target_image.crop((0, 0, w, h))
        
        lowres = target_image.resize((w // scale, h // scale), resample=Image.BICUBIC)
        input_tensor = transform_input(lowres).unsqueeze(0).to(device)
        target_tensor = transform_target(target_image).unsqueeze(0).to(device)

        with torch.inference_mode():
            sr = model(input_tensor).clamp(0, 1)

        psnr_acc += psnr(sr, target_tensor).item()
        ssim_acc += ssim(sr, target_tensor).item()
        
        # There are 2 images that cause lpips to fail
        try:
            x = lpips(sr, target_tensor).cpu().item()
            if np.isnan(x):
                failed_lpips += 1
                continue
                
            lpips_acc += x
        except:
            failed_lpips += 1

    lpips_acc /= len(target_ds) - failed_lpips
    psnr_acc /= len(target_ds)
    ssim_acc /= len(target_ds)
    return psnr_acc, ssim_acc, lpips_acc

In [11]:
def train_step(model, dataloader, optimizer, loss_fn):
    avg_psnr = 0
    avg_ssim = 0
    model.train()

    for batch, target in dataloader:
        batch, target = batch.to(device), target.to(device)
        
        logits = model(batch)
        loss = loss_fn(logits, target)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        logits = logits.clamp(0.0, 1.0)
        target = target.clamp(0.0, 1.0)
        
        avg_psnr += psnr(logits, target).item()
        avg_ssim += ssim(logits, target).item()
        
    avg_psnr /= len(dataloader)
    avg_ssim /= len(dataloader)
    return avg_psnr, avg_ssim

def valid_step(model, dataloader, loss_fn):
    avg_psnr = 0
    avg_ssim = 0
    avg_lpips = 0
    model.eval()

    with torch.inference_mode():
        for batch, target in dataloader:
            batch, target = batch.to(device), target.to(device)

            logits = model(batch)
            
            logits = logits.clamp(0.0, 1.0)
            target = target.clamp(0.0, 1.0)
            
            avg_psnr += psnr(logits, target).item()
            avg_ssim += ssim(logits, target).item()
            avg_lpips += lpips(logits, target).item()

    avg_psnr /= len(dataloader)
    avg_ssim /= len(dataloader)
    avg_lpips /= len(dataloader)

        
    return avg_psnr, avg_ssim, avg_lpips

In [12]:
def train(model, train_dl, valid_dl, optimizer, scheduler: StepLR, loss_fn, epochs, start_checkpoint=None):
    os.makedirs('./tmp_model_checkpoints', exist_ok=True)
    counter = 0 # count epochs without printing training stats
    log_freq = epochs // 20 # how often to print stats when no progress is made
    
    if start_checkpoint:
        start_epoch = start_checkpoint['epoch']
        best_psnr = start_checkpoint['best_psnr']
        best_ssim = start_checkpoint['best_ssim']
        best_lpips = start_checkpoint['best_lpips']
    else:
        start_epoch = 0
        best_psnr = 0
        best_ssim = 0
        best_lpips = float('inf')
        
    for epoch in tqdm(range(start_epoch, epochs), desc="Epochs"):
        counter += 1
        train_psnr, train_ssim = train_step(
            model,
            train_dl,
            optimizer,
            loss_fn
        )

        valid_psnr, valid_ssim, valid_lpips = valid_step(
            model,
            valid_dl,
            loss_fn,
        )

        scheduler.step()

        progress = False
        
        if valid_psnr > best_psnr:
            progress = True
            best_psnr = valid_psnr
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model._orig_mod.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }
            torch.save(checkpoint, f'../tmp_model_checkpoints/best_psnr.pth')

        if valid_ssim > best_ssim:
            progress = True
            best_ssim = valid_ssim
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model._orig_mod.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }
            torch.save(checkpoint, f'../tmp_model_checkpoints/best_ssim.pth')

        if valid_lpips < best_lpips:
            progress = True
            best_lpips = valid_lpips
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model._orig_mod.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }
            torch.save(checkpoint, f'../tmp_model_checkpoints/best_lpips.pth')

        if epoch == epochs-1:
            checkpoint = {
                'epoch': epoch,
                'best_psnr': best_psnr,
                'best_ssim': best_ssim,
                'best_lpips': best_lpips,
                'model_state_dict': model._orig_mod.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }
            torch.save(checkpoint, f'../tmp_model_checkpoints/last.pth')
            
        if progress or counter >= log_freq:
            counter = 0
            print(
                f"Epoch: {epoch+1} | "
                f"learning rate: {scheduler.get_last_lr()[0]:.6f} | "
                f"[train] PSNR: {train_psnr:.4f} | "
                f"[train] SSIM: {train_ssim:.4f} | "
                f"[valid] PSNR: {valid_psnr:.4f} | "
                f"[valid] SSIM: {valid_ssim:.4f} | "
                f"[valid] LPIPS: {valid_lpips:.4f}"
            )

## X2 Scaling

### Training

In [13]:
valid_ds = RCAN_Dataset(HR_valid_paths, 2, ram_limit_gb=1)

Preloading images:   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
train_ds = RCAN_Dataset(HR_train_paths, 2, ram_limit_gb=8)

Preloading images:   0%|          | 0/800 [00:00<?, ?it/s]

#### Day 1

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 2000)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 2

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 4000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 3


In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 6000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 4

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 8000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 5

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 10000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 6

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 12000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 7

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 14000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 8

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 16000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 9

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 18000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Day 10

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = torch.compile(RCAN(2).to(device))
loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-4)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler = StepLR(optimizer, step_size=4000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 20000, checkpoint)

Epochs:   0%|          | 0/2000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.000100 | [train] PSNR: 18.8078 | [train] SSIM: 0.3661 | [valid] PSNR: 21.6358 | [valid] SSIM: 0.5646 | [valid] LPIPS: 0.3644
Epoch: 2 | learning rate: 0.000100 | [train] PSNR: 23.4892 | [train] SSIM: 0.6463 | [valid] PSNR: 24.2203 | [valid] SSIM: 0.6926 | [valid] LPIPS: 0.2630


In [ ]:
model.eval();

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Super-resolution showcase

In [ ]:
print("31px -> 62px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/31px', exist_ok=True)
img.save('../image_results/2X/31px/RCAN_31px.png')
img

In [ ]:
print("63px -> 126px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/63px', exist_ok=True)
img.save('../image_results/2X/63px/RCAN_63px.png')
img

In [ ]:
print("127px -> 254px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/127px', exist_ok=True)
img.save('../image_results/2X/127px/RCAN_127px.png')
img

In [ ]:
print("255px -> 510px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/255px', exist_ok=True)
img.save('../image_results/2X/255px/RCAN_255px.png')
img

In [ ]:
print("510px -> 1020px")
inp = transform(Image.open(X4_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/510px', exist_ok=True)
img.save('../image_results/2X/510px/RCAN_510px.png')
img

In [ ]:
print("1020px -> 2040px")
inp = transform(Image.open(X2_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/1020px', exist_ok=True)
img.save('../image_results/2X/1020px/RCAN_1020px.png')
img

## Geometric Self-Ensemble

The [EDSR paper](https://arxiv.org/pdf/1707.02921) introduced the concept of Geometric Self-Ensemble - referred as **EDSR+**. it supposed to give better results at the cost of **8x** longer inference. The same technique can be applied to RCAN which in the [paper](https://arxiv.org/abs/1807.02758) is refered as **RCAN+**, let's test it.

In [ ]:
class GSE:
    def __init__(self, model):
        self.model = model
        self.rotations = [0, 90, 180, 270]

        self.transforms = v2.Compose([
            v2.PILToTensor(),
            v2.Lambda(lambda x: x / 255.0)
        ])

    def __call__(self, x):
        prediction = torch.zeros(self.model(x).shape).to(device)
        for rotation in self.rotations:
            rot = torch.rot90(x, k=rotation // 90, dims=[-2, -1])
            for i in range(2):
                with torch.inference_mode():
                    if i:
                        flip = v2.functional.horizontal_flip(rot)
                        out = torch.flip(self.model(flip), dims=[-1])
                    else:
                        out = self.model(rot)

                prediction += torch.rot90(out, k=4 - rotation // 90, dims=[-2, -1])

        return prediction / 8.0

In [ ]:
model.eval()
model_gse = GSE(model)

In [17]:
targets = [X32_valid_paths, X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x2 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=6):
    metrics_x2.loc[len(metrics_x2)] = calc_metrics(model_gse, target_ds, 2)

metrics_x2.index = [
    "31px -> 62px", "63px -> 126px", "127px -> 254px", 
    "255px -> 510px", "510px -> 1020px", "1020px -> 2040px"
]
metrics_x2

<All keys matched successfully>

#### Super-resolution showcase

In [ ]:
print("31px -> 62px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model_gse(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/31px', exist_ok=True)
img.save('../image_results/2X/31px/RCAN+_31px.png')
img

In [ ]:
print("63px -> 126px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model_gse(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/63px', exist_ok=True)
img.save('../image_results/2X/63px/RCAN+_63px.png')
img

In [ ]:
print("127px -> 254px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model_gse(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/127px', exist_ok=True)
img.save('../image_results/2X/127px/RCAN+_127px.png')
img

In [ ]:
print("255px -> 510px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model_gse(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/255px', exist_ok=True)
img.save('../image_results/2X/255px/RCAN+_255px.png')
img

In [ ]:
print("510px -> 1020px")
inp = transform(Image.open(X4_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model_gse(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/510px', exist_ok=True)
img.save('../image_results/2X/510px/RCAN+_510px.png')
img

In [ ]:
print("1020px -> 2040px")
inp = transform(Image.open(X2_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model_gse(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/2X/1020px', exist_ok=True)
img.save('../image_results/2X/1020px/RCAN+_1020px.png')
img